# **Crash Dataset**

* link to dataset: https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95/data_preview

In [2]:
import pandas as pd
from helpers import summarize_columns


df = pd.read_csv(
    "datasets/Motor_Vehicle_Collisions_-_Crashes_20260407.csv", delimiter=","
)

summarize_columns(df)

/tmp/ipykernel_71742/2945933504.py:5: DtypeWarning: Columns (0: ZIP CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


                             name    dtype   unique  size (MB)
0                      CRASH DATE      str     5025         38
1                      CRASH TIME      str     1440         27
2                         BOROUGH      str        6         28
3                        ZIP CODE   object      439         77
4                        LATITUDE  float64   130328         17
5                       LONGITUDE  float64   101116         17
6                        LOCATION      str   393219         62
7                  ON STREET NAME      str    23176         65
8               CROSS STREET NAME      str    25238         46
9                 OFF STREET NAME      str   267508         30
10      NUMBER OF PERSONS INJURED  float64       33         17
11       NUMBER OF PERSONS KILLED  float64        8         17
12  NUMBER OF PEDESTRIANS INJURED    int64       14         17
13   NUMBER OF PEDESTRIANS KILLED    int64        6         17
14      NUMBER OF CYCLIST INJURED    int64        5    

#### Person — column reference

<sub>One of three sibling tables: **Person** · [Crash](data_cleaning_crash.ipynb) · [Vehicle](data_cleaning_vehicle.ipynb).&nbsp;&nbsp;
Tags &nbsp; <kbd>PK</kbd> primary key &nbsp; <kbd>FK→C</kbd> foreign key → Crash &nbsp; <kbd>FK→V</kbd> foreign key → Vehicle &nbsp; <kbd>·</kbd> Person-only.</sub>

|  | Column | dtype | key |
|:--|:--|:--|:--|
| **id**     | `UNIQUE_ID`              | `int64 → Int64`                                     | <kbd>PK</kbd> |
|            | `COLLISION_ID`           | `int64 → Int64`                                     | <kbd>FK→C</kbd> |
|            | `VEHICLE_ID`             | `float64 → Int64`                                   | <kbd>FK→V</kbd> |
|            | `PERSON_ID`              | `str → Int64` *factorized*                          | <kbd>·</kbd> |
| **time**   | `CRASH_DATE`             | `str → ✕` *merged*                                 | <kbd>·</kbd> |
|            | `CRASH_TIME`             | `str → ✕` *merged*                                 | <kbd>·</kbd> |
|            | `CRASH_DATETIME`         | `→ datetime64[ns]` *derived*                        | <kbd>·</kbd> |
| **demo**   | `PERSON_AGE`             | `object → Int8` *invalid ages set to `<NA>`*        | <kbd>·</kbd> |
|            | `PERSON_SEX`             | `str → category`                                    | <kbd>·</kbd> |
|            | `PERSON_TYPE`            | `str → category`                                    | <kbd>·</kbd> |
| **injury** | `PERSON_INJURY`          | `str → category`                                    | <kbd>·</kbd> |
|            | `EMOTIONAL_STATUS`       | `str → category`                                    | <kbd>·</kbd> |
|            | `BODILY_INJURY`          | `str → category`                                    | <kbd>·</kbd> |
|            | `COMPLAINT`              | `str → category`                                    | <kbd>·</kbd> |
|            | `EJECTION`               | `str → category`                                    | <kbd>·</kbd> |
| **ped**    | `PED_LOCATION`           | `str → category` *normalized to two values*         | <kbd>·</kbd> |
|            | `PED_ACTION`             | `str → category`                                    | <kbd>·</kbd> |
|            | `PED_ROLE`               | `str → category`                                    | <kbd>·</kbd> |
| **occ**    | `POSITION_IN_VEHICLE`    | `str → category`                                    | <kbd>·</kbd> |
|            | `SAFETY_EQUIPMENT`       | `str → category`                                    | <kbd>·</kbd> |
| **factor** | `CONTRIBUTING_FACTOR_1`  | `str → category`                                    | <kbd>·</kbd> |
|            | `CONTRIBUTING_FACTOR_2`  | `str → category`                                    | <kbd>·</kbd> |

## **Cleaning up section & Formatting**

In [ ]:
# --- Removing Lat/Lon NaN values --- #
_rows_before = df.shape[0]
df = df[df["LONGITUDE"].notna() & df["LATITUDE"].notna()]
print(f"-> Dropped {_rows_before - df.shape[0]} rows after removing lat/lon NaNs rows")


# --- Removing incidents with no injuries or deaths --- #
_rows_before = df.shape[0]
df = df[
    df["NUMBER OF PERSONS INJURED"].notna() | df["NUMBER OF PERSONS KILLED"].notna()
]
print(f"-> Dropped {_rows_before - df.shape[0]} fender bender rows")


# --- Mergin Time Columns in one DateTime column --- #
df["CRASH DATETIME"] = pd.to_datetime(
    df["CRASH DATE"] + " " + df["CRASH TIME"], format="%m/%d/%Y %H:%M"
)

df["CRASH DATETIME"] = pd.to_datetime(
    df["CRASH DATE"], format="%m/%d/%Y"
)

df["CRASH DATETIME"] = pd.to_datetime(df["CRASH TIME"], format="%H:%M"
)

# --- Removing redundant columns --- #
df = df.drop(["LOCATION"], axis=1)


# summarize_columns(df)

-> Dropped 240676 rows after removing lat/lon NaNs rows
-> Dropped 10 fender bender rows


#### <span style="background-color: RebeccaPurple;"> `Fixing missing BUROUGHS` </span>

In [4]:
import geopandas as gpd
from geodatasets import get_path

# -- Title Entries ex: BRONX -> Bronx
df["BOROUGH"] = df["BOROUGH"].str.title()


print(
    "Unique Boroughs include",
    list(df["BOROUGH"].unique()),
    "\n\n\t - Number of rows with missing 'BOROUGH' but present coordinates is:",
    sum(df["BOROUGH"].isna()),
)


# --- Adding geometry column to match and fill missing burough --- #
def _create_missing_geodf(df):
    missing_mask = df["BOROUGH"].isna()
    df_missing = df.loc[missing_mask, ["LONGITUDE", "LATITUDE"]]

    gdf_missing = gpd.GeoDataFrame(
        df_missing,
        geometry=gpd.points_from_xy(
            df_missing["LONGITUDE"], df_missing["LATITUDE"]
        ),  # Function to create the mandatory 'geometry' column for a GeoDataFrame
        crs="EPSG:4326",  # Coordinate Reference System
    )
    return gdf_missing


_gdf_missing = _create_missing_geodf(df)


# --- Using geodatasets to source boroughs multipoligon ---#
def _create_boroughs_lookup():
    path_to_boroughs = get_path("nybb")  # Literally path to the library chache
    boroughs_gdf = gpd.read_file(path_to_boroughs)

    boroughs_gdf = boroughs_gdf.to_crs(
        "EPSG:4326"
    )  # Convert to standard coordinate reference system
    return boroughs_gdf


_boroughs_gdf = _create_boroughs_lookup()


# --- Left spacial join gdf_missing with boroughs multi-poligon ---#
_joined = gpd.sjoin(
    _gdf_missing,
    _boroughs_gdf[["BoroName", "geometry"]],
    how="left",
    predicate="within",
)

df["BOROUGH"] = df["BOROUGH"].fillna(_joined["BoroName"])

print(
    "\n\t - Remaining missing BOROUGH after spatial fill:",
    df["BOROUGH"].isna().sum(),
    "\tFilled :",
    _gdf_missing.shape[0] - df["BOROUGH"].isna().sum(),
)

# --- Removing rows with BOROUGH == nan, after the fill ---#
df = df[df["BOROUGH"].notna()]

# summarize_columns(df)
del _gdf_missing, _boroughs_gdf, _joined

Unique Boroughs include ['Brooklyn', nan, 'Bronx', 'Manhattan', 'Queens', 'Staten Island'] 

	 - Number of rows with missing 'BOROUGH' but present coordinates is: 484639

	 - Remaining missing BOROUGH after spatial fill: 10094 	Filled : 474545


## **Memory Optimization**

In [5]:
_column_set = set(df.columns)

# --- Reducing memory footprint of numberic columns ---#
numeric_columns = [  # Small numeric values
    "NUMBER OF PERSONS INJURED",
    "NUMBER OF PERSONS KILLED",
    "NUMBER OF PEDESTRIANS INJURED",
    "NUMBER OF PEDESTRIANS KILLED",
    "NUMBER OF CYCLIST INJURED",
    "NUMBER OF CYCLIST KILLED",
    "NUMBER OF MOTORIST INJURED",
    "NUMBER OF MOTORIST KILLED",
]

for c in numeric_columns:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int8")


# --- Reducing memory footprint of categorical columns ---#
category_columns = [
    "CONTRIBUTING FACTOR VEHICLE 1",
    "CONTRIBUTING FACTOR VEHICLE 2",
    "CONTRIBUTING FACTOR VEHICLE 3",
    "CONTRIBUTING FACTOR VEHICLE 4",
    "CONTRIBUTING FACTOR VEHICLE 5",
    "VEHICLE TYPE CODE 1",
    "VEHICLE TYPE CODE 2",
    "VEHICLE TYPE CODE 3",
    "VEHICLE TYPE CODE 4",
    "VEHICLE TYPE CODE 5",
]

df = df.astype({k: "category" for k in category_columns})


# --- Reorder columns --- #
new_col_order_list = (
    ["COLLISION_ID", "BOROUGH", "LATITUDE", "LONGITUDE", "CRASH DATETIME"]
    + numeric_columns
    + category_columns
    + ["ON STREET NAME", "CROSS STREET NAME", "OFF STREET NAME"]
)

df = df[new_col_order_list]

assert _column_set == set(_column_set)

summarize_columns(df)

                             name           dtype   unique  size (MB)
0                    COLLISION_ID           int64  2002412         30
1                         BOROUGH             str        5         44
2                        LATITUDE         float64   130276         30
3                       LONGITUDE         float64   101069         30
4                  CRASH DATETIME  datetime64[us]  1184390         30
5       NUMBER OF PERSONS INJURED            Int8       31         19
6        NUMBER OF PERSONS KILLED            Int8        8         19
7   NUMBER OF PEDESTRIANS INJURED            Int8       14         19
8    NUMBER OF PEDESTRIANS KILLED            Int8        6         19
9       NUMBER OF CYCLIST INJURED            Int8        5         19
10       NUMBER OF CYCLIST KILLED            Int8        3         19
11     NUMBER OF MOTORIST INJURED            Int8       29         19
12      NUMBER OF MOTORIST KILLED            Int8        6         19
13  CONTRIBUTING FAC

In [6]:
df = df.reset_index(drop=True)
df.to_parquet("datasets/crash_data.parquet", engine="pyarrow")

In [7]:
pd.Series(df["COLLISION_ID"].unique()).to_csv(
    "datasets/collision_ids.csv", index=False, header=["COLLISION_ID"]
)

In [8]:
del df

---

# **Person Dataset**

* link to dataset: https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Person/f55k-p6yu/data_preview

In [51]:
import pandas as pd
from helpers import summarize_columns

df = pd.read_csv("datasets/Motor_Vehicle_Collisions_-_Person_20260424.csv", sep=",")
collIDs_df = pd.read_csv("datasets/collision_ids.csv", sep=",")

summarize_columns(df)

/tmp/ipykernel_14745/2351309123.py:4: DtypeWarning: Columns (0: PERSON_AGE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("datasets/Motor_Vehicle_Collisions_-_Person_20260424.csv", sep=",")


                     name    dtype   unique  size (MB)
0               UNIQUE_ID    int64  5942295         45
1            COLLISION_ID    int64  1626742         45
2              CRASH_DATE      str     5042        102
3              CRASH_TIME      str     1440         72
4               PERSON_ID      str  5747473        221
5             PERSON_TYPE      str        4         91
6           PERSON_INJURY      str        3        104
7              VEHICLE_ID  float64  2759037         45
8              PERSON_AGE   object     1060        271
9                EJECTION      str        7         78
10       EMOTIONAL_STATUS      str        9         85
11          BODILY_INJURY      str       15         85
12    POSITION_IN_VEHICLE      str       12        117
13       SAFETY_EQUIPMENT      str       18         86
14           PED_LOCATION      str        5         51
15             PED_ACTION      str       17         48
16              COMPLAINT      str       22         91
17        

#### Person — column reference

<sub>One of three sibling tables: **Person** · [Crash](data_cleaning_crash.ipynb) · [Vehicle](data_cleaning_vehicle.ipynb).&nbsp;&nbsp;
Tags &nbsp; <kbd>PK</kbd> primary key &nbsp; <kbd>FK→C</kbd> foreign key → Crash &nbsp; <kbd>FK→V</kbd> foreign key → Vehicle &nbsp; <kbd>·</kbd> Person-only.</sub>

|  | Column | dtype | key |
|:--|:--|:--|:--|
| **id**     | `UNIQUE_ID`              | `int64 → Int64`                                     | <kbd>PK</kbd> |
|            | `COLLISION_ID`           | `int64 → Int64`                                     | <kbd>FK→C</kbd> |
|            | `VEHICLE_ID`             | `float64 → Int64`                                   | <kbd>FK→V</kbd> |
|            | `PERSON_ID`              | `str → Int64` *factorized*                          | <kbd>·</kbd> |
| **time**   | `CRASH_DATE`             | `str → ✕` *merged*                                 | <kbd>·</kbd> |
|            | `CRASH_TIME`             | `str → ✕` *merged*                                 | <kbd>·</kbd> |
|            | `CRASH_DATETIME`         | `→ datetime64[ns]` *derived*                        | <kbd>·</kbd> |
| **demo**   | `PERSON_AGE`             | `object → Int8` *invalid ages set to `<NA>`*        | <kbd>·</kbd> |
|            | `PERSON_SEX`             | `str → category`                                    | <kbd>·</kbd> |
|            | `PERSON_TYPE`            | `str → category`                                    | <kbd>·</kbd> |
| **injury** | `PERSON_INJURY`          | `str → category`                                    | <kbd>·</kbd> |
|            | `EMOTIONAL_STATUS`       | `str → category`                                    | <kbd>·</kbd> |
|            | `BODILY_INJURY`          | `str → category`                                    | <kbd>·</kbd> |
|            | `COMPLAINT`              | `str → category`                                    | <kbd>·</kbd> |
|            | `EJECTION`               | `str → category`                                    | <kbd>·</kbd> |
| **ped**    | `PED_LOCATION`           | `str → category` *normalized to two values*         | <kbd>·</kbd> |
|            | `PED_ACTION`             | `str → category`                                    | <kbd>·</kbd> |
|            | `PED_ROLE`               | `str → category`                                    | <kbd>·</kbd> |
| **occ**    | `POSITION_IN_VEHICLE`    | `str → category`                                    | <kbd>·</kbd> |
|            | `SAFETY_EQUIPMENT`       | `str → category`                                    | <kbd>·</kbd> |
| **factor** | `CONTRIBUTING_FACTOR_1`  | `str → category`                                    | <kbd>·</kbd> |
|            | `CONTRIBUTING_FACTOR_2`  | `str → category`                                    | <kbd>·</kbd> |

#### <span style="background-color: RebeccaPurple;"> `Filtering-in COLLISION IDs present in the Crash Dataset` </span>

In [52]:
df = df[df["COLLISION_ID"].isin(collIDs_df["COLLISION_ID"])]
summarize_columns(df)

                     name    dtype   unique  size (MB)
0               UNIQUE_ID    int64  5344373         81
1            COLLISION_ID    int64  1473702         81
2              CRASH_DATE      str     5025        133
3              CRASH_TIME      str     1440        106
4               PERSON_ID      str  5179194        242
5             PERSON_TYPE      str        4        123
6           PERSON_INJURY      str        3        135
7              VEHICLE_ID  float64  2493335         81
8              PERSON_AGE   object     1005        285
9                EJECTION      str        7        110
10       EMOTIONAL_STATUS      str        9        117
11          BODILY_INJURY      str       15        117
12    POSITION_IN_VEHICLE      str       12        145
13       SAFETY_EQUIPMENT      str       18        117
14           PED_LOCATION      str        5         87
15             PED_ACTION      str       17         84
16              COMPLAINT      str       22        122
17        

## **Cleaning up section & Formatting**

In [ ]:
# --- Person Age --- #
def _clean_person_age(df, col="PERSON_AGE"):
    """Setting ages outside the [0, 120] range to <NA>, allowing for Int8 dtype"""
    df[col] = df[col].str.replace(",", "")
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    mask_invalid = (df[col] < 0) | (df[col] > 120)
    df.loc[mask_invalid, col] = pd.NA

    print(
        f"\t-> {sum(mask_invalid.notna() & mask_invalid)} invalid PERSON_AGE where  set to <NA> "
    )
    return df


df = _clean_person_age(df)


# --- Ped Location i.e. PED_LOCATION ---#
def _binarize_ped_location(df, col="PED_LOCATION"):
    """PED_LOCATION contains both <NA> and 'Unspecified', also naming is a bit too verbose for the entries"""
    d = {
        "Pedestrian/Bicyclist/Other Pedestrian at Intersection": "At Intersection",
        "Pedestrian/Bicyclist/Other Pedestrian Not at Intersection": "Not At Intersection",
    }
    df[col] = df[col].replace(d)

    print(
        f"\t-> {sum(~df[col].isin(d.values()) & df[col].notna())} 'Unspecified' entries were set to <NA>"
    )
    df.loc[~df[col].isin(d.values()), col] = pd.NA

    return df


df = _binarize_ped_location(df)


#! --- Person ID --- #
df["PERSON_ID"] = pd.factorize(df["PERSON_ID"])[0]


df["VEHICLE_ID"] = pd.to_numeric(df["VEHICLE_ID"], errors="coerce").astype("Int64")


# --- Mergin Time Columns in one DateTime column --- #
df["CRASH_DATETIME"] = pd.to_datetime(
    df["CRASH_DATE"] + " " + df["CRASH_TIME"], format="%m/%d/%Y %H:%M"
)

df["CRASH_DATETIME"] = pd.to_datetime(
    df["CRASH_DATE"], format="%m/%d/%Y"
)

df["CRASH_DATETIME"] = pd.to_datetime(df["CRASH_TIME"], format="%H:%M"
)


summarize_columns(df)

	-> 4052 invalid PERSON_AGE where  set to <NA> 
	-> 6734 'Unspecified' entries were set to <NA>
                     name           dtype   unique  size (MB)
0               UNIQUE_ID           int64  5344373         81
1            COLLISION_ID           int64  1473702         81
2               PERSON_ID           int64  5179194         81
3             PERSON_TYPE             str        4        123
4           PERSON_INJURY             str        3        135
5              VEHICLE_ID           Int64  2493335         86
6              PERSON_AGE           Int64      122         86
7                EJECTION             str        7        110
8        EMOTIONAL_STATUS             str        9        117
9           BODILY_INJURY             str       15        117
10    POSITION_IN_VEHICLE             str       12        145
11       SAFETY_EQUIPMENT             str       18        117
12           PED_LOCATION             str        3         83
13             PED_ACTION           

## **Memory Optimization**

In [54]:
_column_set = set(df.columns)


numeric_columns = ["UNIQUE_ID", "COLLISION_ID", "VEHICLE_ID", "PERSON_ID"]

for c in numeric_columns:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

df["PERSON_AGE"] = pd.to_numeric(df["PERSON_AGE"], errors="coerce").astype("Int8")

category_columns = [
    "PERSON_TYPE",
    "PERSON_INJURY",
    "EJECTION",
    "EMOTIONAL_STATUS",
    "BODILY_INJURY",
    "POSITION_IN_VEHICLE",
    "SAFETY_EQUIPMENT",
    "PED_LOCATION",
    "PED_ACTION",
    "COMPLAINT",
    "PED_ROLE",
    "CONTRIBUTING_FACTOR_1",
    "CONTRIBUTING_FACTOR_2",
    "PERSON_SEX",
]

df = df.astype({c: "category" for c in category_columns})


# --- Reorder columns ----
new_col_order_list = (
    numeric_columns + ["CRASH_DATETIME"] + ["PERSON_AGE"] + category_columns
)

assert _column_set == set(_column_set)

summarize_columns(df)

                     name           dtype   unique  size (MB)
0               UNIQUE_ID           Int64  5344373         86
1            COLLISION_ID           Int64  1473702         86
2               PERSON_ID           Int64  5179194         86
3             PERSON_TYPE        category        4         45
4           PERSON_INJURY        category        3         45
5              VEHICLE_ID           Int64  2493335         86
6              PERSON_AGE            Int8      122         50
7                EJECTION        category        7         45
8        EMOTIONAL_STATUS        category        9         45
9           BODILY_INJURY        category       15         45
10    POSITION_IN_VEHICLE        category       12         45
11       SAFETY_EQUIPMENT        category       18         45
12           PED_LOCATION        category        3         45
13             PED_ACTION        category       17         45
14              COMPLAINT        category       22         45
15      

In [55]:
df = df.reset_index(drop=True)
df.to_parquet("datasets/person_data.parquet", engine="pyarrow")

In [56]:
del df

---

# **Vehicle Dataset**

* link to dataset: https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Vehicles/bm4k-52h4/about_data

In [5]:
import pandas as pd
from helpers import summarize_columns

df = pd.read_csv(
    "datasets/Motor_Vehicle_Collisions_-_Vehicles_20260424.csv", delimiter=","
)
collIDs_df = pd.read_csv("datasets/collision_ids.csv")

summarize_columns(df)

/tmp/ipykernel_83281/2781343583.py:4: DtypeWarning: Columns (0: VEHICLE_MODEL, 1: VEHICLE_OCCUPANTS) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


                           name    dtype   unique  size (MB)
0                     UNIQUE_ID    int64  4526815         34
1                  COLLISION_ID    int64  2254964         34
2                    CRASH_DATE      str     5042         77
3                    CRASH_TIME      str     1440         54
4                    VEHICLE_ID      str  2997990        128
5            STATE_REGISTRATION      str       83         43
6                  VEHICLE_TYPE      str     3123        101
7                  VEHICLE_MAKE      str    14690         66
8                 VEHICLE_MODEL      str     2430         35
9                  VEHICLE_YEAR  float64      342         34
10             TRAVEL_DIRECTION      str       16         48
11            VEHICLE_OCCUPANTS   object      204        151
12                   DRIVER_SEX      str        4         37
13        DRIVER_LICENSE_STATUS      str        4         50
14  DRIVER_LICENSE_JURISDICTION      str       73         39
15                    PR

#### Vehicle — column reference

<sub>One of three sibling tables: [Person](data_cleaning_person.ipynb) · [Crash](data_cleaning_crash.ipynb) · **Vehicle**.&nbsp;&nbsp;
Tags &nbsp; <kbd>PK</kbd> primary key &nbsp; <kbd>FK→C</kbd> foreign key → Crash &nbsp; <kbd>·</kbd> Vehicle-only.</sub>

|  | Column | dtype | key |
|:--|:--|:--|:--|
| **id**      | `UNIQUE_ID`                   | `int64 → Int64`                                   | <kbd>PK</kbd> |
|             | `COLLISION_ID`                | `int64 → Int64`                                   | <kbd>FK→C</kbd> |
|             | `VEHICLE_ID`                  | `str → Int64` *factorized*                        | <kbd>·</kbd> |
| **time**    | `CRASH_DATE`                  | `str → datetime64[ns]`                               | <kbd>·</kbd> |
|             | `CRASH_TIME`                  | `str → datetime64[ns]`                                | <kbd>·</kbd> |
|             | `CRASH_DATETIME`              | `→ datetime64[ns]` *derived*                      | <kbd>·</kbd> |
| **vehicle** | `VEHICLE_TYPE`                | `str → category`                                  | <kbd>·</kbd> |
|             | `VEHICLE_MAKE`                | `str → category`                                  | <kbd>·</kbd> |
|             | `VEHICLE_MODEL`               | `str → category`                                  | <kbd>·</kbd> |
|             | `VEHICLE_YEAR`                | `float64 → Int16` *invalid years set to `<NA>`*  | <kbd>·</kbd> |
|             | `VEHICLE_OCCUPANTS`           | `object → Int8` *invalid values set to `<NA>`*   | <kbd>·</kbd> |
|             | `STATE_REGISTRATION`          | `str → category`                                  | <kbd>·</kbd> |
|             | `TRAVEL_DIRECTION`            | `str → category`                                  | <kbd>·</kbd> |
| **driver**  | `DRIVER_SEX`                  | `str → category`                                  | <kbd>·</kbd> |
|             | `DRIVER_LICENSE_STATUS`       | `str → category`                                  | <kbd>·</kbd> |
|             | `DRIVER_LICENSE_JURISDICTION` | `str → category`                                  | <kbd>·</kbd> |
| **crash**   | `PRE_CRASH`                   | `str → category`                                  | <kbd>·</kbd> |
|             | `POINT_OF_IMPACT`             | `str → category`                                  | <kbd>·</kbd> |
| **damage**  | `VEHICLE_DAMAGE`              | `str → category`                                  | <kbd>·</kbd> |
|             | `VEHICLE_DAMAGE_1`            | `str → category`                                  | <kbd>·</kbd> |
|             | `VEHICLE_DAMAGE_2`            | `str → category`                                  | <kbd>·</kbd> |
|             | `VEHICLE_DAMAGE_3`            | `str → category`                                  | <kbd>·</kbd> |
|             | `PUBLIC_PROPERTY_DAMAGE`      | `str → category`                                  | <kbd>·</kbd> |
|             | `PUBLIC_PROPERTY_DAMAGE_TYPE` | `str → category`                                  | <kbd>·</kbd> |
| **factor**  | `CONTRIBUTING_FACTOR_1`       | `str → category`                                  | <kbd>·</kbd> |
|             | `CONTRIBUTING_FACTOR_2`       | `str → category`                                  | <kbd>·</kbd> |

#### <span style="background-color: RebeccaPurple;"> `Filtering-in COLLISION IDs present in the Crash Dataset` </span>

In [6]:
rows_before = df.shape[0]
df = df[df["COLLISION_ID"].isin(collIDs_df["COLLISION_ID"])]
print(f"-> Dropped {rows_before - df.shape[0]} rows not in the crash data")

summarize_columns(df)

-> Dropped 512452 rows not in the crash data
                           name    dtype   unique  size (MB)
0                     UNIQUE_ID    int64  4014363         61
1                  COLLISION_ID    int64  2000806         61
2                    CRASH_DATE      str     5025        100
3                    CRASH_TIME      str     1440         79
4                    VEHICLE_ID      str  2729018        147
5            STATE_REGISTRATION      str       81         68
6                  VEHICLE_TYPE      str     2938        120
7                  VEHICLE_MAKE      str    13746         89
8                 VEHICLE_MODEL      str     1758         61
9                  VEHICLE_YEAR  float64      331         61
10             TRAVEL_DIRECTION      str       16         73
11            VEHICLE_OCCUPANTS   object      190        165
12                   DRIVER_SEX      str        4         63
13        DRIVER_LICENSE_STATUS      str        4         76
14  DRIVER_LICENSE_JURISDICTION      str

## **Cleaning up section & Formatting**

In [7]:
# --- Merging Time Columns into one DateTime column --- #
df["CRASH_DATETIME"] = pd.to_datetime(
    df["CRASH_DATE"] + " " + df["CRASH_TIME"], format="%m/%d/%Y %H:%M"
)
df = df.drop(["CRASH_DATE", "CRASH_TIME"], axis=1)


# --- Factorize VEHICLE_ID --- #
df["VEHICLE_ID"] = pd.factorize(df["VEHICLE_ID"])[0]


# --- Clean VEHICLE_YEAR --- #
df["VEHICLE_YEAR"] = pd.to_numeric(df["VEHICLE_YEAR"], errors="coerce").astype("Int64")
_mask_invalid_year = (df["VEHICLE_YEAR"] < 1900) | (df["VEHICLE_YEAR"] > 2026)
print(f"-> {_mask_invalid_year.sum()} invalid VEHICLE_YEAR set to <NA>")
df.loc[_mask_invalid_year, "VEHICLE_YEAR"] = pd.NA


# --- Clean VEHICLE_OCCUPANTS --- #
df["VEHICLE_OCCUPANTS"] = pd.to_numeric(df["VEHICLE_OCCUPANTS"], errors="coerce").astype("Int64")


summarize_columns(df)

-> 2076 invalid VEHICLE_YEAR set to <NA>
                           name           dtype   unique  size (MB)
0                     UNIQUE_ID           int64  4014363         61
1                  COLLISION_ID           int64  2000806         61
2                    VEHICLE_ID           int64  2729018         61
3            STATE_REGISTRATION             str       81         68
4                  VEHICLE_TYPE             str     2938        120
5                  VEHICLE_MAKE             str    13746         89
6                 VEHICLE_MODEL             str     1758         61
7                  VEHICLE_YEAR           Int64       95         65
8              TRAVEL_DIRECTION             str       16         73
9             VEHICLE_OCCUPANTS           Int64      108         65
10                   DRIVER_SEX             str        4         63
11        DRIVER_LICENSE_STATUS             str        4         76
12  DRIVER_LICENSE_JURISDICTION             str       73         65
13     

## **Memory Optimization**

In [8]:
_column_set = set(df.columns)

numeric_columns = ["UNIQUE_ID", "COLLISION_ID", "VEHICLE_ID"]

for c in numeric_columns:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

df["VEHICLE_YEAR"] = pd.to_numeric(df["VEHICLE_YEAR"], errors="coerce").astype("Int16")

category_columns = [
    "VEHICLE_TYPE",
    "VEHICLE_MAKE",
    "VEHICLE_MODEL",
    "STATE_REGISTRATION",
    "TRAVEL_DIRECTION",
    "DRIVER_SEX",
    "DRIVER_LICENSE_STATUS",
    "DRIVER_LICENSE_JURISDICTION",
    "PRE_CRASH",
    "POINT_OF_IMPACT",
    "VEHICLE_DAMAGE",
    "VEHICLE_DAMAGE_1",
    "VEHICLE_DAMAGE_2",
    "VEHICLE_DAMAGE_3",
    "PUBLIC_PROPERTY_DAMAGE",
    "PUBLIC_PROPERTY_DAMAGE_TYPE",
    "CONTRIBUTING_FACTOR_1",
    "CONTRIBUTING_FACTOR_2",
]

df = df.astype({c: "category" for c in category_columns})


# --- Reorder columns --- #
new_col_order_list = (
    numeric_columns
    + ["CRASH_DATETIME", "VEHICLE_YEAR", "VEHICLE_OCCUPANTS"]
    + category_columns
)

df = df[new_col_order_list]

assert _column_set == set(df.columns)

summarize_columns(df)

                           name           dtype   unique  size (MB)
0                     UNIQUE_ID           Int64  4014363         65
1                  COLLISION_ID           Int64  2000806         65
2                    VEHICLE_ID           Int64  2729018         65
3                CRASH_DATETIME  datetime64[us]  1183880         61
4                  VEHICLE_YEAR           Int16       95         42
5             VEHICLE_OCCUPANTS           Int64      108         65
6                  VEHICLE_TYPE        category     2938         38
7                  VEHICLE_MAKE        category    13746         38
8                 VEHICLE_MODEL        category     1758         38
9            STATE_REGISTRATION        category       81         34
10             TRAVEL_DIRECTION        category       16         34
11                   DRIVER_SEX        category        4         34
12        DRIVER_LICENSE_STATUS        category        4         34
13  DRIVER_LICENSE_JURISDICTION        category 

In [9]:
df = df.reset_index(drop=True)
df.to_parquet("datasets/vehicle_data.parquet", engine="pyarrow")

In [10]:
del df